In [0]:
from pyspark.sql import Row

CATALOG = "power_prices"

# ---------------------------------------------------------------
# 1. dim_border: physical borders between in-scope bidding zones
# ---------------------------------------------------------------
borders = [
    ("AT","CH"),("AT","CZ"),("AT","DE_LU"),("AT","HU"),("AT","IT_NORD"),("AT","SI"),
    ("BE","DE_LU"),("BE","FR"),("BE","NL"),
    ("CH","DE_LU"),("CH","FR"),("CH","IT_NORD"),
    ("CZ","DE_LU"),("CZ","PL"),("CZ","SK"),
    ("DE_LU","DK_1"),("DE_LU","DK_2"),("DE_LU","FR"),("DE_LU","NL"),("DE_LU","NO_2"),
    ("DE_LU","PL"),("DE_LU","SE_4"),
    ("ES","FR"),("ES","PT"),("FR","IT_NORD"),
    ("DK_1","DK_2"),("DK_1","NL"),("DK_1","NO_2"),("DK_1","SE_3"),("DK_2","SE_4"),
    ("NL","NO_2"),
    ("EE","FI"),("EE","LV"),("FI","NO_4"),("FI","SE_1"),("FI","SE_3"),
    ("NO_1","NO_2"),("NO_1","NO_3"),("NO_1","NO_5"),("NO_1","SE_3"),("NO_2","NO_5"),
    ("NO_3","NO_4"),("NO_3","NO_5"),("NO_3","SE_2"),("NO_4","SE_1"),("NO_4","SE_2"),
    ("SE_1","SE_2"),("SE_2","SE_3"),("SE_3","SE_4"),("LT","SE_4"),("PL","SE_4"),
    ("LT","LV"),("LT","PL"),("PL","SK"),
    ("HU","SK"),("HR","HU"),("HU","RO"),("HU","RS"),("HU","SI"),
    ("HR","SI"),("HR","RS"),("IT_NORD","SI"),
    ("BG","RO"),("RO","RS"),("BG","GR"),("BG","RS"),("GR","IT_SUD"),
    ("IT_CNOR","IT_NORD"),("IT_CNOR","IT_CSUD"),("IT_CNOR","IT_SARD"),
    ("IT_CSUD","IT_SUD"),("IT_CSUD","IT_SARD"),("IT_CALA","IT_SUD"),("IT_CALA","IT_SICI"),
]

zones = {r.zone_code: r.country_code for r in
         spark.table(f"{CATALOG}.gold.dim_bidding_zone").select("zone_code", "country_code").collect()}

rows = []
for a, b in borders:
    za, zb = sorted([a, b])
    assert za in zones and zb in zones, f"Unknown zone in border {a}-{b}"
    rows.append(Row(border_key=f"{za}-{zb}", zone_a=za, zone_b=zb,
                    is_internal=(zones[za] == zones[zb])))

spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.gold.dim_border (
  border_key   STRING  NOT NULL COMMENT 'Undirected border id, zones sorted alphabetically, e.g. AT-CH',
  zone_a       STRING  NOT NULL COMMENT 'First zone (alphabetical), FK to dim_bidding_zone',
  zone_b       STRING  NOT NULL COMMENT 'Second zone (alphabetical), FK to dim_bidding_zone',
  is_internal  BOOLEAN NOT NULL COMMENT 'True if both zones are in the same country (e.g. NO_1-NO_2)'
)
COMMENT 'Physical borders between in-scope bidding zones; drives cross-border flow ingestion (two directions each)'
""")
spark.createDataFrame(rows).write.insertInto(f"{CATALOG}.gold.dim_border", overwrite=True)

# ---------------------------------------------------------------
# 2. dim_date: calendar 2019-01-01 to 2027-12-31 with EU DST days
# ---------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.gold.dim_date (
  date_key          INT     NOT NULL COMMENT 'yyyymmdd integer key',
  date              DATE    NOT NULL COMMENT 'Local calendar date',
  year              INT     NOT NULL COMMENT 'Calendar year',
  quarter           INT     NOT NULL COMMENT 'Calendar quarter 1-4',
  month             INT     NOT NULL COMMENT 'Month number 1-12',
  month_name        STRING  NOT NULL COMMENT 'Month name, e.g. January',
  month_short       STRING  NOT NULL COMMENT 'Short month name, e.g. Jan',
  year_month        STRING  NOT NULL COMMENT 'yyyy-MM label for monthly charts',
  week_of_year      INT     NOT NULL COMMENT 'ISO week number',
  day_of_month      INT     NOT NULL COMMENT 'Day of month',
  day_of_week       INT     NOT NULL COMMENT 'ISO day of week, 1 = Monday ... 7 = Sunday',
  day_name          STRING  NOT NULL COMMENT 'Day name, e.g. Monday',
  is_weekend        BOOLEAN NOT NULL COMMENT 'True for Saturday and Sunday',
  season            STRING  NOT NULL COMMENT 'Winter (Dec-Feb), Spring, Summer, Autumn',
  is_dst_change_day BOOLEAN NOT NULL COMMENT 'True on EU clock-change Sundays (last Sunday of March and October)',
  hours_in_day      INT     NOT NULL COMMENT 'Local hours in the day: 23 in March change, 25 in October change, else 24'
)
COMMENT 'Calendar dimension 2019-2027 with seasons and EU daylight saving change days'
""")

spark.sql(f"""
INSERT OVERWRITE {CATALOG}.gold.dim_date
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                AS date_key,
  d                                                      AS date,
  year(d)                                                AS year,
  quarter(d)                                             AS quarter,
  month(d)                                               AS month,
  date_format(d, 'MMMM')                                 AS month_name,
  date_format(d, 'MMM')                                  AS month_short,
  date_format(d, 'yyyy-MM')                              AS year_month,
  weekofyear(d)                                          AS week_of_year,
  dayofmonth(d)                                          AS day_of_month,
  ((dayofweek(d) + 5) % 7) + 1                           AS day_of_week,
  date_format(d, 'EEEE')                                 AS day_name,
  dayofweek(d) IN (1, 7)                                 AS is_weekend,
  CASE WHEN month(d) IN (12, 1, 2) THEN 'Winter'
       WHEN month(d) IN (3, 4, 5)  THEN 'Spring'
       WHEN month(d) IN (6, 7, 8)  THEN 'Summer'
       ELSE 'Autumn' END                                 AS season,
  (month(d) IN (3, 10) AND dayofweek(d) = 1 AND dayofmonth(d) >= 25) AS is_dst_change_day,
  CASE WHEN month(d) = 3  AND dayofweek(d) = 1 AND dayofmonth(d) >= 25 THEN 23
       WHEN month(d) = 10 AND dayofweek(d) = 1 AND dayofmonth(d) >= 25 THEN 25
       ELSE 24 END                                       AS hours_in_day
FROM (SELECT explode(sequence(DATE'2019-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) AS d)
""")

# ---------------------------------------------------------------
# 3. dim_hour_of_day: local hour labels and day parts
# ---------------------------------------------------------------
def day_part(h):
    if h <= 5:   return "Night"
    if h <= 9:   return "Morning"
    if h <= 15:  return "Midday"
    if h <= 16:  return "Afternoon"
    if h <= 20:  return "Evening peak"
    return "Late evening"

hours = [Row(hour=h, hour_label=f"{h:02d}:00", day_part=day_part(h),
             is_solar_window=(10 <= h <= 15), is_evening_peak=(17 <= h <= 20))
         for h in range(24)]

spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.gold.dim_hour_of_day (
  hour             INT     NOT NULL COMMENT 'Local hour 0-23 (start of the hour)',
  hour_label       STRING  NOT NULL COMMENT 'Label for charts, e.g. 13:00',
  day_part         STRING  NOT NULL COMMENT 'Night, Morning, Midday, Afternoon, Evening peak, Late evening',
  is_solar_window  BOOLEAN NOT NULL COMMENT 'True for 10:00-15:59 local, when solar output peaks',
  is_evening_peak  BOOLEAN NOT NULL COMMENT 'True for 17:00-20:59 local, typical demand and price peak'
)
COMMENT 'Local hour-of-day dimension with day parts for solar and peak-hour analysis'
""")
spark.createDataFrame(hours).write.insertInto(f"{CATALOG}.gold.dim_hour_of_day", overwrite=True)

# ---------------------------------------------------------------
# 4. Quality checks
# ---------------------------------------------------------------
b = spark.table(f"{CATALOG}.gold.dim_border")
d = spark.table(f"{CATALOG}.gold.dim_date")
h = spark.table(f"{CATALOG}.gold.dim_hour_of_day")

assert b.count() == b.select("border_key").distinct().count(), "Duplicate border"
assert d.count() == d.select("date_key").distinct().count(), "Duplicate date"
dst = d.filter("is_dst_change_day").count()
assert dst == 18, f"Expected 18 DST change days (2 per year x 9 years), got {dst}"
assert h.count() == 24, "Expected 24 hours"

isolated = (spark.table(f"{CATALOG}.gold.dim_bidding_zone").select("zone_code")
            .subtract(b.select("zone_a").union(b.select("zone_b"))))
print(f"dim_border: {b.count()} borders ({b.filter('is_internal').count()} internal) | "
      f"dim_date: {d.count()} days, {dst} DST days | dim_hour_of_day: {h.count()} hours | checks passed")
print("Zones with no in-scope border:", [r.zone_code for r in isolated.collect()])

display(d.filter("is_dst_change_day").select("date", "day_name", "hours_in_day").orderBy("date").limit(4))